# In-Class Practice: Recursion & Generators

**Task:** Use recursion and generators for common data tasks and compare their strengths.


## 1. Recursive Search

Write a recursive function `find_files(fs_dict, ext)` to find all file paths in a nested dictionary that end with a given extension `ext`.


In [3]:
def find_files(fs_dict, ext, current_path=""):
    """
    Recursively search for files with a specific extension in a nested dictionary.
    
    :param fs_dict: Nested dictionary representing the file system.
    :param ext: The file extension to look for (e.g., '.txt').
    :param current_path: The accumulated path during recursion.
    :return: A list of full file paths matching the extension.
    """
    matched_files = []
    
    for key, value in fs_dict.items():
        # Construct the path for the current node
        node_path = f"{current_path}/{key}" if current_path else key
        
        if isinstance(value, dict):
            # If it's a directory (dictionary), recurse into it
            matched_files.extend(find_files(value, ext, node_path))
        elif isinstance(value, str):
            # If it's a file (string), check the extension
            if value.endswith(ext):
                matched_files.append(f"{node_path}/{value}")
                
    return matched_files

# Test the recursive search
file_system = {
    "home": {
        "user": {
            "docs": {
                "file1": "notes.txt",
                "file2": "image.png",
                "reference": "forReference.py"
            },
            "scripts": {
                "main": "app.py",
                "helper": "utils.py"
            }
        },
        "admin": "config.txt"
    }
}

print("Found '.py' files:")
print(find_files(file_system, ".py"))

print("\nFound '.txt' files:")
print(find_files(file_system, ".txt"))


Found '.py' files:
['home/user/docs/reference/forReference.py', 'home/user/scripts/main/app.py', 'home/user/scripts/helper/utils.py']

Found '.txt' files:
['home/user/docs/file1/notes.txt', 'home/admin/config.txt']


## 2. Memory-Efficient Log Scan

Write a generator function `get_errors(log_path)` to yield only the lines starting with "ERROR" from a potentially huge log file, keeping memory usage low.


In [2]:
import os

# Create a dummy log file for testing
dummy_log_path = "server_logs.txt"
with open(dummy_log_path, "w") as f:
    f.write("INFO: System started\n")
    f.write("ERROR: Connection timeout\n")
    f.write("WARNING: High memory usage\n")
    f.write("ERROR: Database locked\n")
    f.write("INFO: User logged in\n")

def get_errors(log_path):
    """
    Generator function to yield lines starting with 'ERROR' from a file.
    Reads the file line by line to ensure memory efficiency.
    """
    with open(log_path, 'r') as file:
        for line in file:
            if line.startswith("ERROR"):
                yield line.strip()

# Test the generator
print("Extracting errors using generator:")
error_gen = get_errors(dummy_log_path)

for error_line in error_gen:
    print(error_line)
    
# Clean up dummy file
if os.path.exists(dummy_log_path):
    os.remove(dummy_log_path)


Extracting errors using generator:
ERROR: Connection timeout
ERROR: Database locked


## 3. Compare Approaches

**Explain: Why is recursion a natural fit for Task 1?**
Recursion is a natural fit for searching a nested dictionary (file system) because a file system is inherently a recursive, tree-like data structure. Each directory can contain files or other directories, which in turn contain more files or directories. Recursion allows us to write a clean, elegant function that handles a single "node" (directory) and calls itself for any "child nodes" (sub-directories), naturally traversing the entire depth of the tree without needing complex manual stack management.

**Explain: Why would recursion be a poor choice for Task 2, especially with millions of log entries?**
Recursion would be a terrible choice for parsing a massive log file for two main reasons:
1. **Call Stack Limit (Stack Overflow)**: Python has a maximum recursion depth (usually around 1000). If we recursively process a file line-by-line, a file with millions of lines will quickly exceed the recursion limit and crash the program with a `RecursionError`.
2. **Memory Overhead**: Each recursive function call adds a new frame to the call stack, consuming memory. A generator, on the other hand, maintains only a single state in memory and yields one line at a time. This keeps the memory footprint constantly low (O(1) memory complexity), regardless of whether the file has 10 lines or 10 million lines.
